# 🛠️ Setting up the environment
- Install terratorch (restart the environment as prompted)
- Import the necessary python libraries


In [ ]:
!pip install terratorch

In [ ]:
import os
import torch
import random
import numpy as np
from PIL import Image

from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import albumentations as A
from albumentations.pytorch import ToTensorV2

from lightning.pytorch import Trainer
from terratorch.models import EncoderDecoderFactory
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datamodules import GenericNonGeoSegmentationDataModule

# 📊 Import the dataset
## Dubai Aerial Semantic Segmentation Dataset
- Aerial LULC dataset;
- RBG bands;
- 6 LULC classes: *'Building', 'Unpaved Area', 'Road', 'Vegetation', 'Water', 'Unlabeled'*;
- Load the data ➡️ organize files in train/test directories for a training run.

In [ ]:
# Load the dataset from hugging_face: https://huggingface.co/datasets/harshinde/Aerial-Imagery

ds = load_dataset("harshinde/Aerial-Imagery")

In [ ]:
# Organize the training directories and split data into training and testing

base_dir = "/content/demo"

dirs = [
    f"{base_dir}/train/images",
    f"{base_dir}/train/labels",
    f"{base_dir}/test/images",
    f"{base_dir}/test/labels"
]

for d in dirs:
  os.makedirs(d, exist_ok=True)

tiles = list(range(72)) # The dataset is composed of 72 annotated images
random.seed(42)         # For consistency across runs

# Randomly splitting the 72 tiles into training and testing sets
random.shuffle(tiles)
split = int(0.8 * len(tiles))
train_tiles = set(tiles[:split])

### 🧹 Clean the dataset
Remap the classes in a consistent way and save them in the corresponding folders

In [ ]:
# Mapping the PIL RGB
MASTER_MAP = {
    (60, 16, 152):   0,   # Building (#3C1098)
    (132, 41, 246):  1,   # Unpaved Area (#8429F6)
    (110, 193, 228): 2,   # Road (#6EC1E4)
    (254, 221, 58):  3,   # Vegetation (#FEDD3A)
    (226, 169, 41):  4,   # Water (#E2A929)
    (155, 155, 155): 5,   # Unlabeled (#9B9B9B)
    (255, 255, 255): 5    # Unlabeled
}

# Helper function that converts the RGB PIL mask into a the remapped single-channel Image
def rgb_to_class_id(rgb_mask):
    """Converts an RGB PIL image to a single-channel Class ID array."""
    data = np.array(rgb_mask)
    h, w, _ = data.shape
    label_mask = np.zeros((h, w), dtype=np.uint8)

    # Remapping the masks into class_ids based on the RGB values
    for color, class_id in MASTER_MAP.items():
        matching = np.all(data == color, axis=-1)
        label_mask[matching] = class_id

    return Image.fromarray(label_mask, mode='L')

In [ ]:
# Save the images and corresponding masks to the correct directories renaming them appropriately
tile_id = 0

for i in range(0, len(ds['train']), 18):
  images = ds['train'][i:i+9]     # The loaded ds has a structure of alternating 9 images followed by 9 masks.
  masks  = ds['train'][i+9:i+18]

  for j in range(len(images['image'])):
    img = images['image'][j]
    mask_rgb = masks['image'][j].convert('RGB')
    mask = rgb_to_class_id(mask_rgb)

    subset = "train" if tile_id in train_tiles else "test"

    img.save(f"{base_dir}/{subset}/images/tile_{tile_id+1:03d}.png")
    mask.save(f"{base_dir}/{subset}/labels/tile_{tile_id+1:03d}.png")

    tile_id += 1

### 🗺️ Visualize an example from the dataset

In [ ]:
# visualize example
from matplotlib.colors import ListedColormap
vis_img = Image.open("/content/demo/train/images/tile_008.png")
vis_mask = np.array(Image.open("/content/demo/train/labels/tile_008.png"))
classes = ["Building", "Unpaved Area", "Road", "Vegetation", "Water", "Unlabeled"]
colors = ["firebrick","gray","orange","green","blue","black"]
cmap = ListedColormap(colors)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(vis_img)
plt.subplot(1, 2, 2)
plt.imshow(vis_mask, cmap=cmap, vmin=0, vmax=5)
patches = [
    mpatches.Patch(color=cmap(i), label=classes[i])
    for i in range(6)
]
plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.axis("off")
plt.tight_layout()
plt.show()

# 📑 Set up the datamodule

Datamodules are python classes that get your ready to feed to a DL model. TerraTorch offers four Generic Datamodules (Segmentation, Regression, Classification, and Multimodal) and a number of Specific Datamodules for concrete datasets (such as Sen1Floods11, HLS_BurnScars, Landslide4Sense, and more).

To set up a GFM pipeline we need to clearly define a Datamodule for our data (this entails we know the structure and characteristics of our data very well).

In [ ]:
# First we need the statistics of our training data to perform standardization
from terratorch.utils import compute_statistics # TerraTorch provides a built in function to calculate the statistics

# Define a temporary datamodule
temp_dm = GenericNonGeoSegmentationDataModule(
    batch_size=4,
    num_workers=2,
    num_classes=6,
    train_data_root="/content/demo/train/images",
    train_label_data_root="/content/demo/train/labels",
    val_data_root="/content/demo/test/images",
    val_label_data_root="/content/demo/test/labels",
    test_data_root="/content/demo/test/images",
    test_label_data_root="/content/demo/test/labels"
)
temp_dm.setup("fit")

# Compute the statistics
stats = compute_statistics(temp_dm.train_dataloader())
means, stds = stats['means'], stats['stds']

In [ ]:
stds

In [ ]:
# Define the datamodule with the transforms
datamodule = GenericNonGeoSegmentationDataModule(
    batch_size=4,
    num_workers=2,

    num_classes=6,

    train_data_root="/content/demo/train/images",
    train_label_data_root="/content/demo/train/labels",
    val_data_root="/content/demo/test/images",
    val_label_data_root="/content/demo/test/labels",
    test_data_root="/content/demo/test/images",
    test_label_data_root="/content/demo/test/labels",

    train_transform=A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),

        A.Resize(224, 224),
        A.Normalize(mean=means,
                    std=stds,
                    max_pixel_value=1.0),
        ToTensorV2(),
    ]),

    val_transform=A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=means,
                    std=stds,
                    max_pixel_value=1.0),
        ToTensorV2(),
    ]),

    test_transform=A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=means,
                    std=stds,
                    max_pixel_value=1.0),
        ToTensorV2(),
    ])
)

datamodule.setup("fit")

In [ ]:
# We can use the built in .plot() function of the datamodule to visualize the data:

train_dataset = datamodule.train_dataset
train_dataset.plot(train_dataset[5])
plt.show()

# ⚙️ Set up the model

In [ ]:
from terratorch.registry import TERRATORCH_BACKBONE_REGISTRY

# We can check in the Backbone registry what pretrained models we can use;
# In this example we will use one of the Prithvi backbones
# For a detailed overview of Prithvi check out a previous AI4Good Workshop: https://www.youtube.com/watch?v=CB3FKtmuPI8

[backbone
 for backbone in TERRATORCH_BACKBONE_REGISTRY
 if ('prithvi') in backbone]

In [ ]:
# Set up the Backbone, Neck, Decoder parameters:
model_args = dict(
    # Backbone setup: pretrained Prithvi using the RGB bands
    backbone="prithvi_eo_v2_300",
    backbone_pretrained=True,   # Load the pretrained weights
    backbone_num_frames=1,
    backbone_bands=["BLUE", "GREEN", "RED"],

    # Decoder setup:
    decoder="UNetDecoder",
    decoder_channels = [512, 256, 128, 64],

    # Neck setup:
    necks=[{"name": "SelectIndices", "indices": [5, 11, 17, 23]},
           {"name": "ReshapeTokensToImage"},
           {"name": "LearnedInterpolateToPyramidal"}],

    num_classes=6,
    head_dropout=0.1
)

In [ ]:
# Define the TerraTorch task using the model arguments:
task = SemanticSegmentationTask(
    model_args,
    "EncoderDecoderFactory",
    loss="ce",
    lr=1e-4,
    ignore_index=-1,
    optimizer="AdamW",
    optimizer_hparams={"weight_decay": 0.05},
    freeze_backbone = False,
    freeze_decoder = False,
    plot_on_val = False,
    class_names = ["Building", "Unpaved Area", "Road", "Vegetation", "Water", "Unlabeled"],
)

In [ ]:
# Define the Lightning trainer:
trainer = Trainer(accelerator="cuda",
                  max_epochs=5,       # For demo purposes just 5 epochs
                  devices=1,
                  precision='16-mixed',
                  default_root_dir="/content/demo/" # Where the experiments' outputs will be
                  )

# 🧠 Fine-tune the model

In [ ]:
# With everything set up, the training is run calling this one line of code:

trainer.fit(model=task, datamodule=datamodule)

# ✅ Test the fine-tuned model

In [ ]:
best_ckpt_path = "/content/demo/lightning_logs/version_1/checkpoints/epoch=4-step=70.ckpt" # Load the checkpoint
trainer.test(model=task, datamodule=datamodule, ckpt_path=best_ckpt_path)

## 🖼️ Visualize the predictions

In [ ]:
model = SemanticSegmentationTask.load_from_checkpoint(
    best_ckpt_path,
    model_factory=task.hparams.model_factory,
    model_args=task.hparams.model_args,
).cuda() # Ensure the model is on the GPU

test_loader = datamodule.test_dataloader()
test_dataset = datamodule.test_dataset # Ensure test_dataset is available for plotting

with torch.no_grad():
    batch = next(iter(test_loader))
    images = batch["image"].to(model.device) # Move images to the same device as the model
    masks = batch["mask"].numpy()

    with torch.no_grad():
        outputs = model(images)

    preds = torch.argmax(outputs.output, dim=1).cpu().numpy()

for i in range(4):
    sample = {
        "image": batch["image"][i].cpu(),
        "mask": batch["mask"][i],
        "prediction": preds[i],
    }
    test_dataset.plot(sample)
    plt.show()

# 📜 Using the YAML file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pwd

In [ ]:
!terratorch fit -c /content/drive/MyDrive/AI4Good/aerial_dubai.yaml

In [ ]:
!terratorch test -c /content/drive/MyDrive/AI4Good/aerial_dubai.yaml --ckpt_path /content/demo/lightning_logs/version_3/checkpoints/epoch=4-step=70.ckpt